In [17]:
# --- Section 1: Install & Import Libraries ---
!pip install tensorflow pandas scikit-learn nltk --quiet

import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# NLTK import and download stopwords
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords', quiet=True)
_stopwords = set(stopwords.words('english'))

print("Libraries imported and NLTK stopwords downloaded.")


Libraries imported and NLTK stopwords downloaded.


In [18]:
# --- Section 2: Load Dataset ---

df = pd.read_csv('news.csv')  # Read the uploaded file

# Keep only relevant columns
required_columns = ['text', 'label']
if not all(col in df.columns for col in required_columns):
    raise ValueError(f"Dataset must have columns: {required_columns}")

df = df[required_columns]

print("Dataset loaded successfully.")
print(df.head())
print(df['label'].value_counts())


Dataset loaded successfully.
                                                text label
0  Daniel Greenfield, a Shillman Journalism Fello...  FAKE
1  Google Pinterest Digg Linkedin Reddit Stumbleu...  FAKE
2  U.S. Secretary of State John F. Kerry said Mon...  REAL
3  — Kaydee King (@KaydeeKing) November 9, 2016 T...  FAKE
4  It's primary day in New York and front-runners...  REAL
label
REAL    3171
FAKE    3164
Name: count, dtype: int64


In [19]:
# --- Section 3: Preprocessing ---

# Drop missing values
df.dropna(inplace=True)

# Map labels to binary
label_mapping = {'FAKE': 0, 'REAL': 1}
df['label'] = df['label'].map(label_mapping)
df.dropna(subset=['label'], inplace=True)
df['label'] = df['label'].astype(int)

# Function to clean text
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(f'[{re.escape(string.punctuation)}]', '', text)
    text = re.sub(r'\n', '', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    if _stopwords:
        text = ' '.join(word for word in text.split() if word not in _stopwords)
    return text

df['cleaned_text'] = df['text'].apply(clean_text)
print(df['cleaned_text'].head())


0    daniel greenfield shillman journalism fellow f...
1    google pinterest digg linkedin reddit stumbleu...
2    us secretary state john f kerry said monday st...
3    — kaydee king kaydeeking november lesson tonig...
4    primary day new york frontrunners hillary clin...
Name: cleaned_text, dtype: object


In [20]:
# --- Section 4: Train-Test Split & Tokenization ---

X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vocab_size = 20000
embedding_dim = 128

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Dynamic max length (95th percentile)
all_lengths = [len(s) for s in tokenizer.texts_to_sequences(df['cleaned_text'])]
maxlen = int(np.percentile(all_lengths, 95))
maxlen = max(maxlen, 128)  # Ensure reasonable minimum

X_train_pad = pad_sequences(X_train_seq, maxlen=maxlen, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=maxlen, padding='post', truncating='post')

print(f"Padded sequence shape: Train={X_train_pad.shape}, Test={X_test_pad.shape}")


Padded sequence shape: Train=(5068, 1112), Test=(1267, 1112)


In [21]:
# --- Section 5: Build Model ---

model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=maxlen, name='embedding_layer'),
    LSTM(128, name='lstm_layer'),
    Dense(64, activation='relu', name='dense_1'),
    Dropout(0.5, name='dropout_1'),
    Dense(32, activation='relu', name='dense_2'),
    Dropout(0.3, name='dropout_2'),
    Dense(1, activation='sigmoid', name='output_layer')
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_layer (Embedding)     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_layer (LSTM)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
# --- Section 6: Train Model ---
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=16,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 340s 1s/step - accuracy: 0.4941 - loss: 0.6951 - val_accuracy: 0.4892 - val_loss: 0.6932
Epoch 2/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 359s 1s/step - accuracy: 0.5035 - loss: 0.6939 - val_accuracy: 0.4872 - val_loss: 0.6933
Epoch 3/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 317s 1s/step - accuracy: 0.5071 - loss: 0.7167 - val_accuracy: 0.5108 - val_loss: 0.6933


In [23]:
# --- Section 7: Evaluate Model ---

loss, accuracy = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")


Test Loss: 0.6931, Test Accuracy: 0.5051
40/40 ━━━━━━━━━━━━━━━━━━━━ 24s 590ms/step
Precision: 0.7692, Recall: 0.0158, F1-score: 0.0309


In [24]:
# --- Section 8: Predict New Articles ---

test_articles = {
    "Fake News Example": "BREAKING! Secret Alien Base Found In Moon's Tycho Crater! NASA hid evidence...",
    "Real News Example": "U.S. Secretary of State John F. Kerry said Monday that he will stop in Paris..."
}

for name, article in test_articles.items():
    cleaned = clean_text(article)
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=maxlen, padding='post', truncating='post')
    prob = model.predict(pad)[0][0]
    label = 'REAL' if prob > 0.5 else 'FAKE'
    print(f"{name}: Probability={prob:.4f}, Predicted={label}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step
Fake News Example: Probability=0.4984, Predicted=FAKE
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
Real News Example: Probability=0.4984, Predicted=FAKE
